<a href="https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os
if not os.path.exists('/content/flyrank-ml-internship'):
    !git clone https://github.com/him2079/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

!pip install -q duckdb

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.install_extension("httpfs")
con.load_extension("httpfs")
con.execute(f"""CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');""")

/content/flyrank-ml-internship


## 1. My rule and its reason codes

Checking two signals my rule idea leans on: staleness (behind the refresh flags from the session) and CTR-vs-position (behind the CTR-fix logic).

In [9]:
staleness_check = con.execute("""
    SELECT
        CASE
            WHEN days_since = 0 THEN '0-30d'
            WHEN days_since <= 90 THEN '31-90d'
            WHEN days_since <= 180 THEN '91-180d'
            ELSE '180d+'
        END as staleness_bucket,
        COUNT(*) as n,
        AVG(gsc_clicks) as avg_clicks
    FROM (
        SELECT content_hash_id, client_hash_id,
            DATE_DIFF('day', MIN(report_date), MAX(report_date)) as days_since,
            SUM(gsc_clicks) as gsc_clicks
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY 1,2
    )
    GROUP BY 1
    ORDER BY 1
""").df()
staleness_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_clicks
0,0-30d,207,0.000000
1,31-90d,331230,2.481152


In [10]:
ctr_position_check = con.execute("""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN 'pos_1-3'
            WHEN gsc_avg_position <= 10 THEN 'pos_4-10'
            WHEN gsc_avg_position <= 20 THEN 'pos_11-20'
            ELSE 'pos_20+'
        END as position_bucket,
        COUNT(*) as n,
        SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions),0) as ctr
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_impressions > 0
    GROUP BY 1
    ORDER BY 1
""").df()
ctr_position_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,ctr
0,pos_1-3,727362,0.003803
1,pos_11-20,519223,0.003146
2,pos_20+,908354,0.001314
3,pos_4-10,1456122,0.003235


Signal 1 — Staleness verdict: FALSE. Only two buckets appeared (0-30d, n=207; 31-90d, n=331,230), which reveals the query is broken, not that staleness lacks signal — this slice only spans March, so days_since (max/min report_date within the month) can never actually exceed 30 days. It's measuring "days this page appeared in the slice," not true content staleness, which would require joining dim_content.content_created_at. A clearly-broken check is still a useful result: I'm dropping staleness from this baseline rather than build on a measurement that doesn't test what it claims to.

Signal 2 — CTR vs. position verdict: CONFIRMED. CTR drops as position worsens: 0.38% (pos 1-3) → 0.32% (pos 4-10) → 0.31% (pos 11-20) → 0.13% (pos 20+). The relationship holds directionally, with the sharpest drop past position 20 — this is the real signal behind the CTR-fix logic, and it holds in this data.

My rule, in plain words: Flag content with meaningful search visibility (at least 50 impressions) whose click-through rate falls well below the median CTR of other pages at the same position tier — since comparing CTR only within a tier avoids penalizing pages simply for ranking lower, and isolates pages that are genuinely underperforming their own visibility.

Reason codes: low_ctr_for_position (CTR gap > 0.3 relative to tier median), monitor (everything else).

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

Building the score from CTR gap relative to position-tier median (not the global median, which was distorted by a large share of zero-click rows), weighted by log-scaled impressions so high-visibility pages surface first. Only same-window, currently-observable signals are used — no future window, no product flags. Writing the ranked queue to work/outputs/baseline_action_score.csv.

In [12]:
baseline_df = con.execute("""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) as impressions,
        SUM(gsc_clicks) as clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions),0) as ctr
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 50
""").df()

import numpy as np

baseline_df['position_tier'] = np.select(
    [baseline_df['avg_position'] <= 3, baseline_df['avg_position'] <= 10, baseline_df['avg_position'] <= 20],
    ['pos_1-3', 'pos_4-10', 'pos_11-20'],
    default='pos_20+'
)

tier_median_ctr = baseline_df.groupby('position_tier')['ctr'].transform('median')
baseline_df['ctr_gap_score'] = np.clip(1 - (baseline_df['ctr'] / tier_median_ctr), 0, 1)
baseline_df['score'] = baseline_df['ctr_gap_score'] * np.log1p(baseline_df['impressions'])

def reason_code(row):
    if row['ctr_gap_score'] > 0.3:
        return 'low_ctr_for_position'
    else:
        return 'monitor'

baseline_df['reason_code'] = baseline_df.apply(reason_code, axis=1)
baseline_df['action'] = baseline_df['reason_code'].map({
    'low_ctr_for_position': 'rewrite_title_meta',
    'monitor': 'monitor'
})

ranked = baseline_df.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

ranked.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,position_tier,ctr_gap_score,score,reason_code,action
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,pos_4-10,0.995592,11.760848,low_ctr_for_position,rewrite_title_meta
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,pos_4-10,0.995205,11.672405,low_ctr_for_position,rewrite_title_meta
2,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,pos_4-10,0.932770,11.441586,low_ctr_for_position,rewrite_title_meta
3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834.0,1.0,11.195379,0.000012,pos_11-20,0.978823,11.096533,low_ctr_for_position,rewrite_title_meta
4,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,4.0,7.786219,0.000045,pos_4-10,0.973358,11.096402,low_ctr_for_position,rewrite_title_meta
5,client_73cda7b4e4f265ea,content_425715547c6a3ea8,71513.0,3.0,6.395691,0.000042,pos_4-10,0.975040,10.898649,low_ctr_for_position,rewrite_title_meta
6,client_a80fca3f171ed1de,content_046fc480045b88f5,83788.0,6.0,7.289152,0.000072,pos_4-10,0.957392,10.853056,low_ctr_for_position,rewrite_title_meta
7,client_23a62021009f63c4,content_bf078007df823490,44707.0,0.0,7.906249,0.000000,pos_4-10,1.000000,10.707908,low_ctr_for_position,rewrite_title_meta
8,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,pos_4-10,0.917042,10.624877,low_ctr_for_position,rewrite_title_meta
9,client_62f4a7e64f5e0096,content_0c5606abaaab3178,38865.0,0.0,5.694764,0.000000,pos_4-10,1.000000,10.567875,low_ctr_for_position,rewrite_title_meta


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

1.content_8e1334d6356668e3 — action: rewrite_title_meta; reason: 134,984 impressions, 1 click, position 4.5; would be wrong if: this is a tracking/redirect issue, not a copy problem — CTR this low at a good position is unusually extreme.

2.content_fec55986a1868d62 — same pattern as #1 (124,075 impressions, 1 click); same caveat about checking for a technical issue first.

3.content_44f34c0a90047651 — 212,404 impressions, 24 clicks, position 7.3; more plausible genuine underperformer — enough clicks to trust the CTR gap is real, not a tracking artifact.

4.content_9c057b66c30a3abb — 83,834 impressions, 1 click, position 11.2; would be wrong if this page's snippet renders poorly or lacks a compelling meta description rather than being a true demand mismatch.

5.content_cd3d932d4e1c8db0 — 89,332 impressions, 4 clicks, position 7.8; plausible rewrite candidate — low but nonzero clicks support genuine underperformance.

6.content_425715547c6a3ea8 — 71,513 impressions, 3 clicks, position 6.4; same reasoning as #5.

7.content_046fc480045b88f5 — 83,788 impressions, 6 clicks, position 7.3; would be wrong if intent mismatch (page doesn't match what the query wants) rather than a title/meta issue — worth a manual look.

8.content_bf078007df823490 — 44,707 impressions, 0 clicks, position 7.9; zero clicks with real impressions is a strong flag, but also worth checking it isn't a broken or unindexed result.

9.content_f6116743b00afc2d — 107,584 impressions, 15 clicks, position 9.5; most trustworthy pick so far — decent click volume, clear underperformance vs. tier median.

10.content_0c5606abaaab3178 — 38,865 impressions, 0 clicks, position 5.7; same zero-click caveat as #8.

11.content_f0703fc6ae385591 — 63,201 impressions, 2 clicks, position 10.1; would be wrong if this is a low-intent informational page where low CTR is expected regardless of title quality.

12.content_d61fc394d10cba41 — 38,000 impressions, 1 click, position 2.7 (top 3!); this is the most suspicious one — a page ranking #1-3 with essentially zero clicks strongly suggests a technical/rendering problem, not a copy problem.

13.content_9540d884af3e41fd — 82,376 impressions, 11 clicks, position 7.8; reasonably trustworthy pick given the click volume.

14.content_945d6ff91386c817 — 58,278 impressions, 5 clicks, position 6.1; same reasoning.

15.content_23a42776a7009b65 — 28,950 impressions, 0 clicks, position 9.4; zero-click caveat applies again.

16.content_37a6fac676c8cebb — 48,049 impressions, 4 clicks, position 4.4; would be wrong if this page recently changed rank and hasn't accumulated clicks yet (a lag effect, not a real problem).

17.content_713b157e9c77690a — 24,908 impressions, 0 clicks, position 4.0; zero-click, good position — another likely technical-issue candidate rather than pure copy problem.

18.content_39e19a3ec2d95f9d — 42,185 impressions, 4 clicks, position 9.1; would be wrong if seasonal demand dropped for this topic independent of the listing itself.

19.content_c9f840183215651b — 21,519 impressions, 0 clicks, position 9.1; zero-click caveat again.

20.content_22588e765b93dfac — 38,813 impressions, 4 clicks, position 7.7; plausible genuine rewrite candidate given nonzero clicks.



In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

The weakest picks in this top 20 are the zero-click, high-position rows (#8, #10, #12, #15, #17, #19) — a page ranking in the top 10, or even top 3 in #12's case, with literally zero clicks over a full month is an unusual enough pattern that it's more likely a technical or tracking issue (broken snippet, redirect, indexing problem) than a pure title/meta weakness. A stronger version of this rule would separate "zero clicks" from "some clicks but below tier median" into different reason codes, since they likely need different fixes.

Leakage check: this rule uses only gsc_impressions, gsc_clicks, and gsc_avg_position — all observable within the March window, with no future window and no FlyRank product flags (health_score, priority_score, action_type) used anywhere in the scoring or reason-code logic.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.